# 02 — Feature Engineering

Construction des features pour les modèles de détection d'anomalies :
- Rolling statistics (fenêtres glissantes)
- Lag features (décalages temporels)
- Normalisation avec RobustScaler
- Séquences temporelles pour le LSTM-AE
- Analyse de l'impact des features sur les performances

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from pathlib import Path

from anomaly_detection.data.loader import load_csv, generate_synthetic
from anomaly_detection.data.features import (
    add_rolling_features,
    add_lag_features,
    normalize,
    build_sequences,
)
from anomaly_detection.config import settings

## 1. Chargement

In [ ]:
DATA_PATH = Path('../data/raw/creditcard.csv')

if DATA_PATH.exists():
    df, y = load_csv(DATA_PATH, 'Class')
else:
    df, y = generate_synthetic(n_normal=5000, n_anomaly=100, save=False)

X_raw = df.select_dtypes(include='number')
print(f'Shape brut : {X_raw.shape}')

## 2. Rolling statistics

In [ ]:
# Fenêtres configurées dans settings
print(f'Fenêtres rolling : {settings.rolling_windows}')

X_rolling = add_rolling_features(X_raw)
print(f'Shape après rolling : {X_rolling.shape} (+{X_rolling.shape[1] - X_raw.shape[1]} features)')

In [ ]:
# Visualiser l'effet du rolling sur une feature
feature = X_raw.columns[0]
fig = go.Figure()
fig.add_trace(go.Scatter(y=X_raw[feature][:200], name='Original', line=dict(color='#888780')))
for w in settings.rolling_windows:
    fig.add_trace(go.Scatter(
        y=X_rolling[f'{feature}_roll{w}_mean'][:200],
        name=f'Rolling mean w={w}',
    ))
fig.update_layout(title=f'Rolling means sur "{feature}" (200 premiers points)', height=400)
fig.show()

## 3. Lag features

In [ ]:
print(f'Lags configurés : {settings.lag_periods}')

X_lagged = add_lag_features(X_raw)
print(f'Shape après lags : {X_lagged.shape} (+{X_lagged.shape[1] - X_raw.shape[1]} features)')

## 4. Pipeline complet + normalisation

In [ ]:
# Pipeline complet pour les modèles tabulaires
X_enriched = add_rolling_features(add_lag_features(X_raw))
print(f'Shape final enrichi : {X_enriched.shape}')

X = X_enriched.values.astype(np.float32)
y_np = y.values.astype(np.int32)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_np, test_size=0.2, random_state=settings.seed, stratify=y_np
)
X_train_normal = X_train[y_train == 0]

X_train_s, X_test_s, scaler = normalize(X_train_normal, X_test)
print(f'Train (normaux seulement) : {X_train_normal.shape}')
print(f'Test : {X_test.shape} — {y_test.sum()} anomalies')

In [ ]:
# Vérifier la distribution après normalisation
fig = go.Figure()
for i in range(min(5, X_train_s.shape[1])):
    fig.add_trace(go.Box(y=X_train_s[:, i], name=X_enriched.columns[i], boxmean=True))
fig.update_layout(
    title='Distribution après RobustScaler (5 premières features)',
    height=400,
    yaxis_range=[-5, 5],
)
fig.show()

## 5. Séquences pour LSTM-AE

In [ ]:
# Construction des séquences temporelles
seq_len = settings.lstm_seq_len
print(f'Longueur de séquence : {seq_len}')

# On utilise les features brutes normalisées pour le LSTM
X_raw_norm = scaler.transform(X_raw.values.astype(np.float32))
sequences = build_sequences(X_raw_norm[:500], seq_len=seq_len)  # 500 premiers points
print(f'Shape séquences : {sequences.shape}  → (n_samples, seq_len, n_features)')

In [ ]:
# Visualiser une séquence
seq_idx = 42
fig = px.imshow(
    sequences[seq_idx].T,
    labels={'x': 'Pas de temps', 'y': 'Feature', 'color': 'Valeur'},
    title=f'Séquence #{seq_idx} — heatmap (seq_len={seq_len})',
    color_continuous_scale='RdBu_r',
    zmin=-3, zmax=3,
    height=350,
    aspect='auto',
)
fig.show()

## 6. Résumé du pipeline

| Étape | Input | Output | Modèles concernés |
|-------|-------|--------|------------------|
| Features brutes | `(n, p)` | `(n, p)` | tous |
| + Rolling | `(n, p)` | `(n, p×13)` | IF, AE |
| + Lags | `(n, p)` | `(n, p×4)` | IF, AE |
| RobustScaler | `(n, d)` | `(n, d)` | tous |
| Séquences | `(n, d)` | `(n-L+1, L, d)` | LSTM-AE |